# Knowledge-Base Ingestion — AST-Based Approach



In [50]:
# Install required packages (run once)
# %pip install markdown-it-py pyyaml pydantic langchain-core

In [51]:
from __future__ import annotations

from pathlib import Path
from typing import Literal, Optional

import yaml
from markdown_it import MarkdownIt
from pydantic import BaseModel, Field, ValidationError
from langchain_core.documents import Document

## Frontmatter Schema (Pydantic)

Validates every field on ingest — a misspelled `status` or `policy_authority` raises a
`ValidationError` instead of silently becoming `"unknown"`.


In [52]:
class PolicyFrontMatter(BaseModel):
    document_id: str
    title: str
    status: Literal["active", "superseded", "draft"]
    policy_authority: Literal["official", "unofficial"]
    audience: str = "all"
    supersedes: Optional[str] = None
    superseded_by: Optional[str] = None

    @property
    def is_citable_authority(self) -> bool:
        return self.status == "active" and self.policy_authority == "official"

## Frontmatter Extraction

Simple manual split on `---` fences — avoids importing a whole frontmatter library for one job.


In [53]:
def split_frontmatter(raw_text: str) -> tuple[dict, str]:
    """Extract YAML frontmatter from a markdown file."""
    if not raw_text.startswith("---"):
        return {}, raw_text
    parts = raw_text.split("---", 2)
    if len(parts) < 3:
        return {}, raw_text
    _, fm_raw, body = parts
    return (yaml.safe_load(fm_raw) or {}), body.lstrip("\n")

## AST-Based Section Splitting

Walks **markdown-it-py**'s token stream to group content under each heading.
Tracks a full heading path so nested sections (e.g. h2 inside h1) stay distinguishable.
Avoids false splits on `##` inside code fences or inline text.


In [54]:
def split_by_headings(body: str, max_level: int = 3) -> list[dict]:
    """
    Walk the markdown token stream and group content under each heading,
    tracking a full heading path (e.g. ["Refund Policy", "Exceptions"]).
    Returns a list of {"heading_path": [...], "text": "..."} dicts.
    """
    md = MarkdownIt()
    tokens = md.parse(body)

    sections: list[dict] = []
    heading_stack: list[tuple[int, str]] = []  # (level, text)
    current_lines: list[str] = []
    lines = body.splitlines()

    def flush(path_snapshot):
        text = "\n".join(current_lines).strip()
        if text:
            sections.append({"heading_path": list(path_snapshot), "text": text})

    i = 0
    line_cursor = 0
    while i < len(tokens):
        tok = tokens[i]
        if tok.type == "heading_open":
            level = int(tok.tag[1])  # h1 -> 1, h2 -> 2, etc.
            inline_tok = tokens[i + 1]
            heading_text = inline_tok.content

            # flush content accumulated under the OLD heading path
            flush([h for _, h in heading_stack])
            current_lines.clear()

            # pop stack back to this level, then push the new heading
            heading_stack = [(lvl, txt) for lvl, txt in heading_stack if lvl < level]
            if level <= max_level:
                heading_stack.append((level, heading_text))

            i += 3  # skip heading_open, inline, heading_close
            continue

        if tok.type == "inline" and tok.map:
            start, end = tok.map
            current_lines.extend(lines[line_cursor:end] if end > line_cursor else [])
            line_cursor = max(line_cursor, end)

        i += 1

    flush([h for _, h in heading_stack])

    if not sections:
        sections = [{"heading_path": [], "text": body.strip()}]

    return sections

## Knowledge-Base Loader

Produces `Document` objects ready for a text splitter / vector store.
Files that fail Pydantic validation are **skipped** and reported as errors — 
a policy doc with invalid metadata should never enter the index silently.


In [55]:
def load_knowledge_base(kb_dir: str | Path) -> tuple[list[Document], list[str]]:
    """Load all .md files from kb_dir, validate frontmatter, split by headings."""
    kb_dir = Path(kb_dir)
    documents: list[Document] = []
    load_errors: list[str] = []

    for filepath in sorted(kb_dir.glob("**/*.md")):
        raw_text = filepath.read_text(encoding="utf-8")
        fm_dict, body = split_frontmatter(raw_text)

        try:
            fm = PolicyFrontMatter(**fm_dict)
        except ValidationError as e:
            # Fail loudly — skip this file rather than silently guessing values
            load_errors.append(f"{filepath.name}: {e}")
            continue

        sections = split_by_headings(body)

        for idx, section in enumerate(sections):
            heading_path = section["heading_path"]
            section_text = section["text"]
            if len(section_text) < 3:
                continue

            heading_label = " > ".join(heading_path) if heading_path else fm.title
            # prefix chunk text with its location so it's meaningful in isolation
            content_with_context = f"{fm.title} > {heading_label}\n\n{section_text}"

            documents.append(Document(
                page_content=content_with_context,
                metadata={
                    "chunk_id": f"{filepath.stem}::{idx}",
                    "filename": filepath.name,
                    "document_id": fm.document_id,
                    "title": fm.title,
                    "heading_path": heading_label,
                    "status": fm.status,
                    "policy_authority": fm.policy_authority,
                    "audience": fm.audience,
                    "supersedes": fm.supersedes,
                    "superseded_by": fm.superseded_by,
                    "is_citable_authority": fm.is_citable_authority,
                },
            ))

    return documents, load_errors

## Load the Knowledge Base

Point the loader at the `knowledge-base/` directory (the **primary data source** — unchanged).


In [56]:
BASE_DIR = Path.cwd()
if not (BASE_DIR / "knowledge-base").is_dir():
    BASE_DIR = BASE_DIR.parent
KNOWLEDGE_BASE = BASE_DIR / "knowledge-base"

docs, errors = load_knowledge_base(KNOWLEDGE_BASE)

print(f"Loaded {len(docs)} chunks from {KNOWLEDGE_BASE}\n")
for d in docs:
    tag = "AUTHORITY" if d.metadata["is_citable_authority"] else "no-authority"
    print(f"[{tag:12}] {d.metadata['filename']:40} {d.metadata['heading_path']}")

if errors:
    print(f"\n{len(errors)} file(s) FAILED validation and were skipped:")
    for e in errors:
        print("   -", e)

Loaded 49 chunks from c:\Users\hp\Desktop\AI\genAI-basics\ai-agent-test\knowledge-base

[AUTHORITY   ] 01-returns-policy-current.md             Returns Policy > Standard return window
[AUTHORITY   ] 01-returns-policy-current.md             Returns Policy > Item condition
[AUTHORITY   ] 01-returns-policy-current.md             Returns Policy > Return shipping and refunds
[AUTHORITY   ] 01-returns-policy-current.md             Returns Policy > Exclusions and exceptions
[no-authority] 02-returns-policy-legacy.md              Returns Policy — Legacy Version
[no-authority] 02-returns-policy-legacy.md              Returns Policy — Legacy Version > Return window
[no-authority] 02-returns-policy-legacy.md              Returns Policy — Legacy Version > Return shipping
[no-authority] 02-returns-policy-legacy.md              Returns Policy — Legacy Version > Condition requirements
[no-authority] 02-returns-policy-legacy.md              Returns Policy — Legacy Version > Refund timing
[AUTHORITY   

## Inspect a Sample Document


In [57]:
if docs:
    sample = docs[0]
    print("=== page_content ===")
    print(sample.page_content[:500])
    print("\n=== metadata ===")
    for k, v in sample.metadata.items():
        print(f"  {k}: {v}")

=== page_content ===
Returns Policy > Returns Policy > Standard return window

# Returns Policy

## Standard return window

Customers on the standard plan may request a return within **30 calendar days of delivery**.

TrailPlus members receive a different return window. See the TrailPlus Membership Policy. The membership must have been active when the order was placed.

=== metadata ===
  chunk_id: 01-returns-policy-current::0
  filename: 01-returns-policy-current.md
  document_id: RET-2026-01
  title: Returns Policy
  heading_path: Returns Policy > Standard return window
  status: active
  policy_authority: official
  audience: customer
  supersedes: RET-2024-01
  superseded_by: None
  is_citable_authority: True


## Embedding & Vector Store

Use the loaded `docs` (which already have rich metadata) with an embedding model and Chroma.


In [58]:
from langchain_chroma.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.embeddings import init_embeddings

In [59]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""]
)
chunks = text_splitter.split_documents(docs)

print(f"Chunks after text splitting: {len(chunks)}")

Chunks after text splitting: 54


In [60]:
embedding = init_embeddings(
    "huggingface:BAAI/bge-small-en-v1.5")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4359.55it/s]


In [61]:
vector_store = Chroma.from_documents(
    collection_name="example_collection",
    documents=chunks,
    embedding=embedding,
    persist_directory="./chroma_langchain_db",
)

## Evaluation Queries


In [62]:
questions = [
    "How many days do standard customers have to return an item, and is there a return shipping fee?",
    "What was the old return window and return shipping policy for orders placed before April 2026?",
    "What extra return and shipping benefits do TrailPlus members get on US orders?",
    "Can I get assistance or a replacement if an item marked FINAL SALE arrives damaged or broken?",
    "Can I cancel my order or change my shipping address after checking out?",
    "What is the warranty period for backpacks versus drinkware, and does Aster & Row have a lifetime warranty?",
    "What is the order amount needed for free standard US shipping, and how long does order processing take before dispatch?",
    "Is the Breeze Tumbler dishwasher safe, and can I put it in the microwave?",
    "Within how many days can I request a price adjustment if an item goes on sale, and what items are excluded?",
    "How quickly must arrival damage be reported, and what happens if a defect is found after that window?",
]
for i in questions:
    print(i)

How many days do standard customers have to return an item, and is there a return shipping fee?
What was the old return window and return shipping policy for orders placed before April 2026?
What extra return and shipping benefits do TrailPlus members get on US orders?
Can I get assistance or a replacement if an item marked FINAL SALE arrives damaged or broken?
Can I cancel my order or change my shipping address after checking out?
What is the warranty period for backpacks versus drinkware, and does Aster & Row have a lifetime warranty?
What is the order amount needed for free standard US shipping, and how long does order processing take before dispatch?
Is the Breeze Tumbler dishwasher safe, and can I put it in the microwave?
Within how many days can I request a price adjustment if an item goes on sale, and what items are excluded?
How quickly must arrival damage be reported, and what happens if a defect is found after that window?


In [69]:
# Create the retriever once with your search parameters
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

for question in questions:
    print(f"\nQuestion: {question}")
    
    # Use invoke() to fetch the relevant documents
    results = retriever.invoke(question)
    
    for doc in results:
        print(f"* {doc.page_content[:200]}...")
        print(f"   metadata: {doc.metadata}")
        print()


Question: How many days do standard customers have to return an item, and is there a return shipping fee?
* Returns Policy > Returns Policy > Return shipping and refunds

## Return shipping and refunds

A **$6.95 return shipping fee** is deducted from the refund for standard domestic returns. The fee is wai...
   metadata: {'chunk_id': '01-returns-policy-current::2', 'heading_path': 'Returns Policy > Return shipping and refunds', 'supersedes': 'RET-2024-01', 'document_id': 'RET-2026-01', 'title': 'Returns Policy', 'status': 'active', 'is_citable_authority': True, 'audience': 'customer', 'policy_authority': 'official', 'filename': '01-returns-policy-current.md'}

* ## Return shipping and refunds

A **$6.95 return shipping fee** is deducted from the refund for standard domestic returns. The fee is waived when Aster & Row sent the wrong item or the item arrived da...
   metadata: {'source': 'c:\\Users\\hp\\Desktop\\AI\\genAI-basics\\ai-agent-test\\knowledge-base\\01-returns-policy-curren

In [71]:
# 1. Define the retriever configuration
retriever = vector_store.as_retriever(search_kwargs={"k": 4})

# 2. Invoke it with your specific query string
results = retriever.invoke("what is the standard return window for laptop ?")

for doc in results:
    print(f"* {doc.page_content[:200]}...")
    print(f"   metadata: {doc.metadata}")
    print()

* Returns Policy — Legacy Version > Returns Policy — Legacy Version > Return window

## Return window

Customers could return eligible merchandise within **45 calendar days of delivery**....
   metadata: {'title': 'Returns Policy — Legacy Version', 'superseded_by': 'RET-2026-01', 'policy_authority': 'official', 'heading_path': 'Returns Policy — Legacy Version > Return window', 'is_citable_authority': False, 'filename': '02-returns-policy-legacy.md', 'document_id': 'RET-2024-01', 'status': 'superseded', 'chunk_id': '02-returns-policy-legacy::1', 'audience': 'customer'}

* Returns Policy > Returns Policy > Standard return window

# Returns Policy

## Standard return window

Customers on the standard plan may request a return within **30 calendar days of delivery**.

Tra...
   metadata: {'policy_authority': 'official', 'filename': '01-returns-policy-current.md', 'supersedes': 'RET-2024-01', 'chunk_id': '01-returns-policy-current::0', 'document_id': 'RET-2026-01', 'is_citable_authority': T